# Detector de armas v3 — ajuste fino com hard negatives

Objetivo: melhorar o detector atual de armas/objetos perigosos, principalmente os falsos positivos em pessoas/roupas/braços e falsos negativos em armas pequenas.

Modelo inicial esperado: `best_v2.pt`.

Classes preservadas:

```text
0 machete
1 knife
2 baseball_bat
3 rifle
4 gun
```

Estratégia:

1. baixar novamente o Dangerous Items Dataset;
2. adicionar Guns-Knives Object Detection do Kaggle;
3. adicionar COCO como negativos difíceis e reforço para `knife` e `baseball_bat`;
4. opcionalmente aceitar um ZIP manual de reforço no formato YOLO;
5. treinar `weapons_v3` sem sobrescrever v2;
6. comparar `best_v2.pt` × `best_v3.pt`;
7. testar imagens externas;
8. exportar ONNX/TFLite e baixar um pacote final.

Meta operacional, não promessa: buscar `mAP50 >= 0.90` e `recall médio >= 0.85`. Se o dataset externo continuar ruim, o notebook mostra isso em vez de mascarar com limiar baixo.

In [ ]:
!pip -q install ultralytics==8.4.67 kagglehub pycocotools pyyaml tqdm pandas opencv-python-headless onnx onnxruntime

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
from collections import Counter, defaultdict
from google.colab import files
from IPython.display import display, Image as IPImage
from pycocotools.coco import COCO
from tqdm.auto import tqdm

import json
import os
import random
import shutil
import subprocess
import zipfile
import yaml

import cv2
import kagglehub
import numpy as np
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO

ultralytics.check()
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Ative GPU T4 no Colab.')

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class Configuracao:
    epocas: int = 60
    tamanho_imagem: int = 768
    lote: int = 12
    paciencia: int = 14
    workers: int = 2
    seed: int = 42

    lr0: float = 0.00045
    lrf: float = 0.03
    freeze: int = 0

    usar_kaggle_guns_knives: bool = True
    usar_coco_hard_negatives: bool = True
    usar_zip_reforco_manual: bool = False

    max_coco_negativos_treino: int = 1800
    max_coco_negativos_val: int = 300
    max_coco_negativos_teste: int = 300
    max_coco_positivos_por_classe: int = 1200

    conf_validacao: float = 0.25
    iou_validacao: float = 0.50
    conf_teste_externo: float = 0.25
    imgsz_teste_externo: int = 960

    exportar: bool = True
    exportar_int8: bool = False
    fracao_calibracao_int8: float = 0.20

CFG = Configuracao()
CLASSES = {0:'machete', 1:'knife', 2:'baseball_bat', 3:'rifle', 4:'gun'}
NOME_PARA_ID = {v:k for k,v in CLASSES.items()}

RAIZ = Path('/content/armas_v3')
ORIG = RAIZ/'dataset_original'
KAGGLE = RAIZ/'dataset_kaggle'
COCO_DIR = RAIZ/'coco'
REFORCO = RAIZ/'reforco_manual'
UNIFICADO = RAIZ/'dataset_unificado'
RESULTADOS = RAIZ/'resultados'
ENTREGA = RAIZ/'entrega'
PESO_INICIAL = RAIZ/'best_v2.pt'

for p in [RAIZ, ORIG, KAGGLE, COCO_DIR, REFORCO, UNIFICADO, RESULTADOS, ENTREGA]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)

display(pd.DataFrame([asdict(CFG)]))

## 2. Enviar o peso inicial `best_v2.pt`

In [ ]:
enviados = files.upload()
pts = [nome for nome in enviados if nome.lower().endswith('.pt')]
if len(pts) != 1:
    raise RuntimeError('Envie exatamente um arquivo .pt: o best_v2.pt.')
shutil.copy2(pts[0], PESO_INICIAL)
modelo_base = YOLO(str(PESO_INICIAL))
print('Peso inicial:', PESO_INICIAL)
print('Classes do peso:', modelo_base.names)
if {int(k): v for k, v in modelo_base.names.items()} != CLASSES:
    raise RuntimeError('As classes do peso não correspondem às classes esperadas.')

## 3. Baixar Dangerous Items Dataset

In [ ]:
URL_DANGEROUS = 'https://zenodo.org/records/16422779/files/Dangerous%20Items.zip?download=1'
ZIP_DANGEROUS = RAIZ/'dangerous_items.zip'

if not ZIP_DANGEROUS.exists():
    subprocess.run(['wget','-c',URL_DANGEROUS,'-O',str(ZIP_DANGEROUS)], check=True)

if not (ORIG/'.extraido').exists():
    with zipfile.ZipFile(ZIP_DANGEROUS) as z:
        z.extractall(ORIG)
    (ORIG/'.extraido').touch()
print('Dangerous Items extraído em:', ORIG)

## 4. Baixar reforço Kaggle para gun/knife

In [ ]:
if CFG.usar_kaggle_guns_knives:
    try:
        caminho_kaggle = Path(kagglehub.dataset_download('iqmansingh/guns-knives-object-detection'))
        print('Guns-Knives:', caminho_kaggle)
    except Exception as e:
        caminho_kaggle = None
        print('Falha ao baixar Kaggle Guns-Knives. O treinamento continuará sem ele:', e)
else:
    caminho_kaggle = None
    print('Reforço Kaggle desativado.')

## 5. Baixar COCO val2017 para positivos e negativos difíceis

COCO é usado aqui de duas formas:

- positivos adicionais para `knife` e `baseball_bat`;
- negativos difíceis com pessoas, celulares, mochilas, roupas e objetos comuns.

Imagens que contêm `knife` ou `baseball bat` recebem rótulo positivo. Imagens sem essas classes, mas com pessoas/objetos comuns, entram com `.txt` vazio.

In [ ]:
if CFG.usar_coco_hard_negatives:
    COCO_IMG_ZIP = COCO_DIR/'val2017.zip'
    COCO_ANN_ZIP = COCO_DIR/'annotations_trainval2017.zip'
    if not COCO_IMG_ZIP.exists():
        subprocess.run(['wget','-c','http://images.cocodataset.org/zips/val2017.zip','-O',str(COCO_IMG_ZIP)], check=True)
    if not COCO_ANN_ZIP.exists():
        subprocess.run(['wget','-c','http://images.cocodataset.org/annotations/annotations_trainval2017.zip','-O',str(COCO_ANN_ZIP)], check=True)
    if not (COCO_DIR/'val2017').exists():
        with zipfile.ZipFile(COCO_IMG_ZIP) as z:
            z.extractall(COCO_DIR)
    if not (COCO_DIR/'annotations').exists():
        with zipfile.ZipFile(COCO_ANN_ZIP) as z:
            z.extractall(COCO_DIR)
    print('COCO pronto:', COCO_DIR)
else:
    print('COCO desativado.')

## 6. Reforço manual opcional

Estrutura aceita:

```text
reforco.zip
├── images/train, images/val, images/test
└── labels/train, labels/val, labels/test
```

Negativos difíceis devem ter `.txt` vazio correspondente.

In [ ]:
if CFG.usar_zip_reforco_manual:
    enviados = files.upload()
    zips = [nome for nome in enviados if nome.lower().endswith('.zip')]
    if len(zips) != 1:
        raise RuntimeError('Envie exatamente um ZIP de reforço.')
    if REFORCO.exists():
        shutil.rmtree(REFORCO)
    REFORCO.mkdir(parents=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(REFORCO)
    print('Reforço manual extraído:', REFORCO)
else:
    print('Reforço manual desativado.')

## 7. Funções de organização e remapeamento YOLO

In [ ]:
EXT_IMG = {'.jpg','.jpeg','.png','.bmp','.webp'}

def imagens_em(p: Path):
    if not p or not p.exists(): return []
    return sorted(x for x in p.rglob('*') if x.suffix.lower() in EXT_IMG)

def criar_dirs_base():
    for split in ['train','val','test']:
        (UNIFICADO/'images'/split).mkdir(parents=True, exist_ok=True)
        (UNIFICADO/'labels'/split).mkdir(parents=True, exist_ok=True)

criar_dirs_base()

def copiar_ou_linkar(src: Path, dst: Path):
    if dst.exists(): return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def label_por_imagem(img: Path):
    partes = list(img.parts)
    if 'images' in partes:
        idx = len(partes) - 1 - partes[::-1].index('images')
        partes[idx] = 'labels'
        return Path(*partes).with_suffix('.txt')
    return img.with_suffix('.txt')

def escrever_label(dst: Path, linhas):
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text('
'.join(linhas).strip() + ('
' if linhas else ''), encoding='utf-8')

def copiar_dataset_yolo(imagens, split, prefixo, mapa_classes=None):
    n_img = 0; n_obj = 0; n_neg = 0
    for img in tqdm(imagens, desc=f'{prefixo}-{split}'):
        out_img = UNIFICADO/'images'/split/f'{prefixo}_{img.stem}{img.suffix.lower()}'
        out_lbl = UNIFICADO/'labels'/split/f'{prefixo}_{img.stem}.txt'
        copiar_ou_linkar(img.resolve(), out_img)
        lbl = label_por_imagem(img)
        linhas_saida = []
        if lbl.exists():
            for linha in lbl.read_text(encoding='utf-8', errors='ignore').splitlines():
                partes = linha.strip().split()
                if len(partes) != 5: continue
                try:
                    cid = int(float(partes[0]))
                except Exception:
                    continue
                novo = mapa_classes.get(cid, cid) if mapa_classes else cid
                if novo not in CLASSES: continue
                linhas_saida.append(' '.join([str(novo)] + partes[1:]))
        escrever_label(out_lbl, linhas_saida)
        n_img += 1; n_obj += len(linhas_saida); n_neg += int(len(linhas_saida)==0)
    return {'fonte':prefixo,'split':split,'imagens':n_img,'objetos':n_obj,'negativas':n_neg}

def encontrar_yaml(pasta: Path):
    for yml in list(pasta.rglob('*.yaml')) + list(pasta.rglob('*.yml')):
        try:
            d = yaml.safe_load(yml.read_text(encoding='utf-8'))
            if isinstance(d, dict) and {'train','val','names'} <= set(d):
                return yml, d
        except Exception:
            pass
    return None, None

def nomes_para_dict(nomes):
    if isinstance(nomes, list): return {i:str(v) for i,v in enumerate(nomes)}
    return {int(k):str(v) for k,v in nomes.items()}

def norm(s): return str(s).strip().lower().replace(' ','_').replace('-','_')

def resolver(valor, base):
    p = Path(str(valor))
    return p if p.is_absolute() else (base/p)

## 8. Montar dataset unificado

In [ ]:
relatorios = []

# Dangerous Items
raiz_danger = ORIG/'Dangerous Items' if (ORIG/'Dangerous Items').exists() else ORIG
p_train = next((p for p in [raiz_danger/'train'/'images', raiz_danger/'Train'/'images', raiz_danger/'train'] if p.exists()), None)
p_val = next((p for p in [raiz_danger/'valid'/'images', raiz_danger/'val'/'images', raiz_danger/'Valid'/'images', raiz_danger/'valid', raiz_danger/'val'] if p.exists()), None)
p_test = next((p for p in [raiz_danger/'test'/'images', raiz_danger/'Test'/'images', raiz_danger/'test'] if p.exists()), None)
if not (p_train and p_val):
    raise RuntimeError(f'Não encontrei train/val no Dangerous Items. Raiz: {raiz_danger}')
relatorios.append(copiar_dataset_yolo(imagens_em(p_train), 'train', 'danger'))
relatorios.append(copiar_dataset_yolo(imagens_em(p_val), 'val', 'danger'))
if p_test:
    relatorios.append(copiar_dataset_yolo(imagens_em(p_test), 'test', 'danger'))

# Kaggle gun/knife
if caminho_kaggle is not None:
    yml, data = encontrar_yaml(caminho_kaggle)
    if yml:
        base = Path(data.get('path',''))
        if not base.is_absolute(): base = yml.parent/base
        names = nomes_para_dict(data['names'])
        mapa = {}
        for cid, nome in names.items():
            n = norm(nome)
            if n in ['gun','pistol','handgun','firearm']:
                mapa[cid] = NOME_PARA_ID['gun']
            elif n in ['knife','knives']:
                mapa[cid] = NOME_PARA_ID['knife']
        if mapa:
            for split, key in [('train','train'),('val','val'),('test','test')]:
                if key in data:
                    imgs = imagens_em(resolver(data[key], base))
                    if imgs:
                        relatorios.append(copiar_dataset_yolo(imgs, split, f'gk_{split}', mapa))
        else:
            print('Kaggle baixado, mas não encontrei classes gun/knife no YAML:', names)
    else:
        print('Kaggle sem YAML reconhecido. Ignorado para evitar rótulos errados.')

pd.DataFrame(relatorios)

## 9. Adicionar COCO: knife/baseball_bat + hard negatives

In [ ]:
if CFG.usar_coco_hard_negatives:
    ann = COCO_DIR/'annotations'/'instances_val2017.json'
    coco = COCO(str(ann))
    cats = {c['name']: c['id'] for c in coco.loadCats(coco.getCatIds())}
    positivos = {'knife': NOME_PARA_ID['knife'], 'baseball bat': NOME_PARA_ID['baseball_bat']}
    ids_pos = [cats[k] for k in positivos if k in cats]
    ids_excluir_neg = set(ids_pos)
    imgs_com_pos = set()
    for cid in ids_pos:
        imgs_com_pos.update(coco.getImgIds(catIds=[cid]))
    imgs_todos = set(coco.getImgIds())

    rng = random.Random(CFG.seed)
    ids_pos_lista = list(imgs_com_pos)
    rng.shuffle(ids_pos_lista)
    ids_neg_lista = list(imgs_todos - imgs_com_pos)
    rng.shuffle(ids_neg_lista)

    def split_lista(lista):
        n=len(lista); a=int(n*0.75); b=int(n*0.88)
        return {'train':lista[:a], 'val':lista[a:b], 'test':lista[b:]}

    pos_splits = split_lista(ids_pos_lista[:CFG.max_coco_positivos_por_classe*2])
    max_neg = {'train':CFG.max_coco_negativos_treino, 'val':CFG.max_coco_negativos_val, 'test':CFG.max_coco_negativos_teste}
    neg_splits = split_lista(ids_neg_lista[:sum(max_neg.values())])
    neg_splits = {k:v[:max_neg[k]] for k,v in neg_splits.items()}

    rel_coco=[]
    for split, ids in pos_splits.items():
        for img_id in tqdm(ids, desc=f'coco-pos-{split}'):
            info = coco.loadImgs([img_id])[0]
            src = COCO_DIR/'val2017'/info['file_name']
            out_img = UNIFICADO/'images'/split/f'coco_pos_{img_id}.jpg'
            out_lbl = UNIFICADO/'labels'/split/f'coco_pos_{img_id}.txt'
            copiar_ou_linkar(src.resolve(), out_img)
            anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id], catIds=ids_pos, iscrowd=False))
            linhas=[]
            for a in anns:
                cat_name = coco.loadCats([a['category_id']])[0]['name']
                if cat_name not in positivos: continue
                x,y,w,h = a['bbox']
                cx=(x+w/2)/info['width']; cy=(y+h/2)/info['height']; ww=w/info['width']; hh=h/info['height']
                if ww<=0 or hh<=0: continue
                linhas.append(f"{positivos[cat_name]} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}")
            escrever_label(out_lbl, linhas)
            rel_coco.append({'fonte':'coco_pos','split':split,'imagens':1,'objetos':len(linhas),'negativas':0})
    for split, ids in neg_splits.items():
        for img_id in tqdm(ids, desc=f'coco-neg-{split}'):
            info = coco.loadImgs([img_id])[0]
            src = COCO_DIR/'val2017'/info['file_name']
            out_img = UNIFICADO/'images'/split/f'coco_neg_{img_id}.jpg'
            out_lbl = UNIFICADO/'labels'/split/f'coco_neg_{img_id}.txt'
            copiar_ou_linkar(src.resolve(), out_img)
            escrever_label(out_lbl, [])
            rel_coco.append({'fonte':'coco_neg','split':split,'imagens':1,'objetos':0,'negativas':1})
    relatorios += rel_coco

pd.DataFrame(relatorios).groupby(['fonte','split']).sum(numeric_only=True).reset_index()

## 10. Adicionar reforço manual, se existir

In [ ]:
if CFG.usar_zip_reforco_manual:
    raiz = REFORCO
    raiz_candidata = next((p.parent for p in raiz.rglob('images') if (p.parent/'labels').exists()), raiz)
    for split in ['train','val','test']:
        imgs = imagens_em(raiz_candidata/'images'/split)
        if imgs:
            relatorios.append(copiar_dataset_yolo(imgs, split, 'manual'))
else:
    print('Sem reforço manual.')

## 11. Auditar dataset unificado

In [ ]:
resumo=[]; classes_count=Counter()
for split in ['train','val','test']:
    imgs = imagens_em(UNIFICADO/'images'/split)
    n_obj=0; n_neg=0; invalidos=0
    for img in imgs:
        lbl = UNIFICADO/'labels'/split/f'{img.stem}.txt'
        if not lbl.exists():
            invalidos += 1
            continue
        linhas=[l.strip() for l in lbl.read_text().splitlines() if l.strip()]
        n_neg += int(len(linhas)==0)
        for l in linhas:
            p=l.split()
            if len(p)!=5: invalidos+=1; continue
            cid=int(float(p[0])); classes_count[(split,CLASSES.get(cid,'INVALID'))]+=1; n_obj+=1
    resumo.append({'split':split,'imagens':len(imgs),'objetos':n_obj,'negativas':n_neg,'invalidos':invalidos})

auditoria=pd.DataFrame(resumo)
display(auditoria)
display(pd.DataFrame([{'split':s,'classe':c,'objetos':n} for (s,c),n in classes_count.items()]).pivot(index='classe',columns='split',values='objetos').fillna(0).astype(int))
if auditoria['invalidos'].sum()>0:
    raise RuntimeError('Há rótulos inválidos ou ausentes no dataset unificado.')

yaml_v3 = RAIZ/'dados_armas_v3.yaml'
yaml_v3.write_text(yaml.safe_dump({'path':str(UNIFICADO), 'train':'images/train', 'val':'images/val', 'test':'images/test', 'names':CLASSES}, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(yaml_v3.read_text())
auditoria.to_csv(RESULTADOS/'auditoria_dataset_v3.csv', index=False)

## 12. Treinar weapons_v3

In [ ]:
modelo = YOLO(str(PESO_INICIAL))
resultado = modelo.train(
    data=str(yaml_v3),
    epochs=CFG.epocas,
    imgsz=CFG.tamanho_imagem,
    batch=CFG.lote,
    patience=CFG.paciencia,
    workers=CFG.workers,
    device=0,
    project=str(RESULTADOS),
    name='weapons_v3',
    exist_ok=True,
    optimizer='AdamW',
    lr0=CFG.lr0,
    lrf=CFG.lrf,
    seed=CFG.seed,
    deterministic=True,
    amp=True,
    cos_lr=True,
    freeze=CFG.freeze,
    mosaic=0.40,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=8,
    plots=True,
    save=True,
    save_period=5,
    verbose=True,
)
PASTA_TREINO = RESULTADOS/'weapons_v3'
BEST_V3 = PASTA_TREINO/'weights'/'best.pt'
LAST_V3 = PASTA_TREINO/'weights'/'last.pt'
print('BEST_V3:', BEST_V3, BEST_V3.exists())

## 13. Comparar best_v2 × best_v3

In [ ]:
modelos = {'v2': YOLO(str(PESO_INICIAL)), 'v3': YOLO(str(BEST_V3))}
comparacao=[]; por_classe=[]
for nome, m in modelos.items():
    met = m.val(data=str(yaml_v3), split='test', imgsz=CFG.tamanho_imagem, batch=CFG.lote, conf=CFG.conf_validacao, iou=CFG.iou_validacao, device=0, project=str(RESULTADOS), name=f'validacao_{nome}', exist_ok=True, plots=True, verbose=True)
    comparacao.append({'modelo':nome,'mAP50-95':float(met.box.map),'mAP50':float(met.box.map50),'mAP75':float(met.box.map75),'precisao_media':float(np.mean(met.box.p)),'revocacao_media':float(np.mean(met.box.r))})
    for i in range(len(met.box.maps)):
        por_classe.append({'modelo':nome,'classe':met.names[i], 'precisao':float(met.box.p[i]), 'revocacao':float(met.box.r[i]), 'mAP50':float(met.box.ap50[i]), 'mAP50-95':float(met.box.maps[i])})
comparacao_df=pd.DataFrame(comparacao)
classe_df=pd.DataFrame(por_classe)
display(comparacao_df)
display(classe_df)
comparacao_df.to_csv(RESULTADOS/'comparacao_v2_v3.csv', index=False)
classe_df.to_csv(RESULTADOS/'metricas_por_classe_v3.csv', index=False)

meta_ok = (comparacao_df.query("modelo=='v3'")['mAP50'].iloc[0] >= 0.90) and (comparacao_df.query("modelo=='v3'")['revocacao_media'].iloc[0] >= 0.85)
print('Meta mAP50>=0.90 e recall>=0.85:', meta_ok)

## 14. Teste externo

Envie imagens como `faca.png`, `faca2.png`, `arma.png`, `arma2.png`, `normal.png`, `normal2.png` e outras imagens reais.

In [ ]:
enviados = files.upload()
PASTA_EXT = RAIZ/'testes_externos'
PASTA_EXT.mkdir(exist_ok=True)
imgs_ext=[]
for nome in enviados:
    p=Path(nome)
    if p.suffix.lower() in EXT_IMG:
        dst=PASTA_EXT/p.name
        shutil.copy2(p,dst)
        imgs_ext.append(dst)
if not imgs_ext:
    raise RuntimeError('Envie ao menos uma imagem externa.')
print('Imagens externas:', len(imgs_ext))

In [ ]:
PRED_EXT = RESULTADOS/'testes_externos'
PRED_EXT.mkdir(exist_ok=True)
linhas=[]
detalhes=[]
for nome_modelo, m in modelos.items():
    resultados = m.predict(source=[str(x) for x in imgs_ext], imgsz=CFG.imgsz_teste_externo, conf=CFG.conf_teste_externo, iou=CFG.iou_validacao, device=0, verbose=False)
    pasta = PRED_EXT/nome_modelo
    pasta.mkdir(exist_ok=True)
    for img, res in zip(imgs_ext, resultados, strict=True):
        dets=[]
        if res.boxes is not None:
            for box, conf, cls in zip(res.boxes.xyxy.cpu().tolist(), res.boxes.conf.cpu().tolist(), res.boxes.cls.cpu().tolist(), strict=True):
                dets.append({'classe':res.names[int(cls)], 'confianca':float(conf), 'caixa':[float(v) for v in box]})
        cv2.imwrite(str(pasta/f'{img.stem}_detectado.jpg'), res.plot())
        linhas.append({'modelo':nome_modelo,'imagem':img.name,'quantidade_deteccoes':len(dets),'classes':', '.join(d['classe'] for d in dets),'maior_confianca':max([d['confianca'] for d in dets], default=0)})
        detalhes.append({'modelo':nome_modelo,'imagem':img.name,'deteccoes':dets})
resumo_ext=pd.DataFrame(linhas)
display(resumo_ext)
resumo_ext.to_csv(RESULTADOS/'resumo_testes_externos_v3.csv', index=False)
(RESULTADOS/'detalhes_testes_externos_v3.json').write_text(json.dumps(detalhes, ensure_ascii=False, indent=2), encoding='utf-8')
for nome_modelo in modelos:
    print('
MODELO', nome_modelo)
    for img in sorted((PRED_EXT/nome_modelo).glob('*.jpg'))[:12]:
        display(IPImage(filename=str(img)))

## 15. Exportar e baixar pacote final

In [ ]:
exportados=[]
if CFG.exportar:
    melhor = YOLO(str(BEST_V3))
    onnx = melhor.export(format='onnx', imgsz=CFG.tamanho_imagem, simplify=True, nms=True, batch=1)
    exportados.append(Path(onnx))
    tf_float = melhor.export(format='tflite', imgsz=CFG.tamanho_imagem, int8=False, half=False, nms=True, batch=1)
    exportados.append(Path(tf_float))
    if CFG.exportar_int8:
        tf_int8 = melhor.export(format='tflite', imgsz=CFG.tamanho_imagem, int8=True, data=str(yaml_v3), fraction=CFG.fracao_calibracao_int8, nms=True, batch=1)
        exportados.append(Path(tf_int8))
    print('Exportados:', exportados)
else:
    print('Exportação desativada.')

if ENTREGA.exists(): shutil.rmtree(ENTREGA)
ENTREGA.mkdir(parents=True)
for src,name in [(PESO_INICIAL,'best_v2_inicial.pt'),(BEST_V3,'best_v3.pt'),(LAST_V3,'last_v3.pt'),(yaml_v3,'dados_armas_v3.yaml')]:
    if Path(src).exists(): shutil.copy2(src, ENTREGA/name)
for f in ['comparacao_v2_v3.csv','metricas_por_classe_v3.csv','auditoria_dataset_v3.csv','resumo_testes_externos_v3.csv','detalhes_testes_externos_v3.json']:
    src=RESULTADOS/f
    if src.exists(): shutil.copy2(src, ENTREGA/f)
for f in ['results.csv','results.png','confusion_matrix.png','confusion_matrix_normalized.png','PR_curve.png','F1_curve.png','P_curve.png','R_curve.png']:
    src=PASTA_TREINO/f
    if src.exists(): shutil.copy2(src, ENTREGA/f)
if PRED_EXT.exists(): shutil.copytree(PRED_EXT, ENTREGA/'testes_externos', dirs_exist_ok=True)
for e in exportados:
    if e.exists():
        if e.is_dir(): shutil.copytree(e, ENTREGA/e.name, dirs_exist_ok=True)
        else: shutil.copy2(e, ENTREGA/e.name)
resumo={'configuracao':asdict(CFG), 'classes':CLASSES, 'comparacao':comparacao_df.to_dict('records'), 'auditoria':auditoria.to_dict('records')}
(ENTREGA/'resumo_experimento_v3.json').write_text(json.dumps(resumo, ensure_ascii=False, indent=2), encoding='utf-8')
zip_final = shutil.make_archive('/content/armas_v3_entrega', 'zip', root_dir=str(ENTREGA))
print('ZIP:', zip_final)
files.download(zip_final)

## Observação crítica

Se `faca2.png` continuar criando caixa em uma pessoa, não insista em mais épocas. A solução será aumentar o reforço manual com negativos difíceis anotados como `.txt` vazio.